# SEAS 6505 — HW 8  
## Toward New Insights on Job Trends: Digital/AI Demand vs Supply Dashboard  
**Student:** Christopher Paul  
**Date:** December 16, 2025  

### Problem statement  
Build an interactive dashboard showing where digital/AI job demand and supply are rising or lagging across countries, industries, and skill types, and conclude with actionable recommendations for decision makers.  

### Context (why this matters)  
Over the next few years, a large wave of young workers will enter labor markets—many in countries where work remains informal and vulnerable to disruption. Digitalization can create better jobs and productivity, but it can also displace existing work. Leaders need near real-time signals to see where the digital/AI transition is accelerating and where people are being left behind.  

### Data sources used (official/public)  
- World Bank World Development Indicators (WDI) via Indicators API (connectivity + digital economy proxies)  
- World Bank Data360 / ILO Employment (ILO_EMP) for ICT employment (supply proxy)  
- World Bank JOIN / Global Jobs Indicators Database (labor market structure & workforce diagnostics)  
- World Bank Global Findex (digital inclusion and access)  
- Digital Progress and Trends Report 2025: Strengthening AI Foundations (context + “four Cs” framing)  

> Note: some sources provide interactive downloads. The code below uses direct APIs/CSV endpoints where available.


## Systems engineering approach (requirements → design)

### What “demand vs supply” means in this homework
Because globally consistent AI job posting microdata is not always openly downloadable, this notebook uses a practical, reproducible approach:

**Digital/AI demand index (country-year):** a composite of:
- Growth in ICT services exports (proxy for digital economy pull)
- Growth in high-technology exports (proxy for tech-intensive production)
- Internet adoption and broadband penetration (proxy for market readiness to absorb digital work)

**Digital/AI supply index (country-year):** a composite of:
- ICT employment (ICT services + ICT manufacturing)
- Tertiary enrollment (proxy for skills pipeline)

### Dashboard views
1) Gap map: countries colored by (Demand Index − Supply Index)  
2) Quadrant plot: Demand vs Supply (“Priority upskilling”, “Scale demand”, etc.)  
3) Trend drill-down: per-country trajectories (2015–latest)  
4) Export HTML: shareable single-file dashboard

### Output artifacts
- Clean merged dataset (CSV)
- Plotly dashboard exported to HTML
- Written story + recommendations


In [ ]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import plotly.express as px

pd.set_option("display.max_columns", 200)


## Scope (chosen to be globally balanced)
High-income: USA, GBR, DEU, KOR  
Upper-middle: CHN, BRA, MEX  
Lower-middle: IND, VNM, NGA  


In [ ]:
countries = {
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "KOR": "Korea, Rep.",
    "CHN": "China",
    "BRA": "Brazil",
    "MEX": "Mexico",
    "IND": "India",
    "VNM": "Vietnam",
    "NGA": "Nigeria",
}

START_YEAR = 2015
END_YEAR = 2024


## Step 1 — Pull WDI indicators (connectivity + digital economy demand proxies)

Indicators:
- IT.NET.USER.ZS Internet users (% of population)  
- IT.NET.BBND.P2 Fixed broadband (per 100 people)  
- BX.GSR.CCIS.ZS ICT service exports (% of service exports)  
- TX.VAL.TECH.CD High-technology exports (US$)  
- SE.TER.ENRR Tertiary enrollment (% gross)  
- SP.POP.TOTL Population (for per-capita conversion)  


In [ ]:
WDI_BASE = "https://api.worldbank.org/v2"

wdi_indicators = {
    "internet_users_pct": "IT.NET.USER.ZS",
    "fixed_broadband_per100": "IT.NET.BBND.P2",
    "ict_service_exports_pct": "BX.GSR.CCIS.ZS",
    "high_tech_exports_usd": "TX.VAL.TECH.CD",
    "tertiary_enroll_gross_pct": "SE.TER.ENRR",
    "population": "SP.POP.TOTL",
}

def fetch_wdi(country_iso3: str, indicator: str) -> pd.DataFrame:
    url = f"{WDI_BASE}/country/{country_iso3}/indicator/{indicator}"
    params = {"format": "json", "per_page": 20000}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    rows = []
    for obs in data[1] if len(data) > 1 and data[1] else []:
        year = int(obs["date"])
        if START_YEAR <= year <= END_YEAR:
            rows.append({"iso3": country_iso3, "year": year, "value": obs["value"]})
    return pd.DataFrame(rows)

def build_wdi_panel() -> pd.DataFrame:
    panel = None
    for iso3 in countries.keys():
        for col, ind in wdi_indicators.items():
            df = fetch_wdi(iso3, ind).rename(columns={"value": col})
            panel = df if panel is None else panel.merge(df, on=["iso3","year"], how="outer")
    panel["country"] = panel["iso3"].map(countries)
    return panel.sort_values(["iso3","year"]).reset_index(drop=True)

wdi = build_wdi_panel()
wdi.head(10)


## Step 2 — Pull ICT employment (Data360 / ILO_EMP)

If the direct URLs work in your environment:
- https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_MAN.csv  
- https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_SER.csv  

If not, download from Data360 manually and load locally.


In [ ]:
def fetch_csv(url: str) -> pd.DataFrame:
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))

ILO_ICT_MAN_URL = "https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_MAN.csv"
ILO_ICT_SER_URL = "https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_SER.csv"

ict_man = fetch_csv(ILO_ICT_MAN_URL)
ict_ser = fetch_csv(ILO_ICT_SER_URL)

ict_man.head()


## Step 3 — Clean, merge, and compute indices

Demand Index (mean z-score):
- ICT service exports %, High-tech exports per capita, Internet users %, Broadband per 100

Supply Index (mean z-score):
- ICT employment (services + manufacturing), Tertiary enrollment

Gap Index = Demand − Supply


In [ ]:
df = wdi.copy()

def tidy_ict(df_raw: pd.DataFrame, value_name: str) -> pd.DataFrame:
    cols = {c.lower(): c for c in df_raw.columns}
    econ = cols.get("economy code") or cols.get("economy_code") or cols.get("economycode") or cols.get("country code")
    time = cols.get("time") or cols.get("year")
    val = cols.get("value") or cols.get("data") or cols.get("obs_value")
    out = df_raw[[econ, time, val]].copy()
    out.columns = ["iso3", "year", value_name]
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out = out.dropna(subset=["iso3","year"])
    out["year"] = out["year"].astype(int)
    return out[(out["year"] >= START_YEAR) & (out["year"] <= END_YEAR)]

ict_man_t = tidy_ict(ict_man, "ict_mfg_emp")
ict_ser_t = tidy_ict(ict_ser, "ict_srv_emp")

df = df.merge(ict_man_t, on=["iso3","year"], how="left")
df = df.merge(ict_ser_t, on=["iso3","year"], how="left")

df["high_tech_exports_per_capita"] = df["high_tech_exports_usd"] / df["population"]

def zscore(s: pd.Series) -> pd.Series:
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or np.isnan(sd) or sd == 0:
        return s * np.nan
    return (s - mu) / sd

demand_components = ["ict_service_exports_pct","high_tech_exports_per_capita","internet_users_pct","fixed_broadband_per100"]
supply_components = ["ict_srv_emp","ict_mfg_emp","tertiary_enroll_gross_pct"]

for c in demand_components + supply_components:
    df[f"z_{c}"] = zscore(df[c])

df["demand_index"] = df[[f"z_{c}" for c in demand_components]].mean(axis=1, skipna=True)
df["supply_index"] = df[[f"z_{c}" for c in supply_components]].mean(axis=1, skipna=True)
df["gap_index"] = df["demand_index"] - df["supply_index"]

df.head(10)


## Step 4 — Visuals + export dashboard to HTML


In [ ]:
latest_year = int(df["year"].max())
latest = df[df["year"] == latest_year].copy()

fig_map = px.choropleth(
    latest,
    locations="iso3",
    color="gap_index",
    hover_name="country",
    hover_data={"demand_index": True, "supply_index": True, "gap_index": True, "iso3": False},
    title=f"Digital/AI Demand–Supply Gap Index (Latest year: {latest_year})",
)

fig_scatter = px.scatter(
    latest,
    x="supply_index",
    y="demand_index",
    text="iso3",
    hover_name="country",
    title=f"Demand vs Supply (Latest year: {latest_year})",
)

fig_trend = px.line(
    df.sort_values(["country","year"]),
    x="year",
    y=["demand_index","supply_index","gap_index"],
    color="country",
    title="Trends: Demand, Supply, Gap (2015–latest)",
)

fig_map.show()
fig_scatter.show()
fig_trend.show()

from plotly.offline import plot

html_parts = [plot(fig, include_plotlyjs='cdn', output_type='div') for fig in [fig_map, fig_scatter, fig_trend]]
html = f"""<html><head><meta charset='utf-8'><title>HW8 Dashboard</title></head>
<body><h1>HW8 — Digital/AI Demand vs Supply Dashboard</h1>
<p><b>Student:</b> Christopher Paul &nbsp;&nbsp; <b>Latest year:</b> {latest_year}</p>
{''.join(html_parts)}
</body></html>"""

with open("HW8_dashboard.html","w",encoding="utf-8") as f:
    f.write(html)

df.to_csv("HW8_demand_supply_dataset.csv", index=False)

("HW8_dashboard.html","HW8_demand_supply_dataset.csv")


## Submission narrative (paste into your write-up)

### Key insights
- Digital demand rises fastest where connectivity and digital trade capacity are already strong.  
- Supply lags where ICT employment and tertiary pipelines are thin, even if internet use is improving.  
- The most urgent policy target is the “high demand / low supply” quadrant: these countries face skill shortages, rising wage pressure, and a risk that AI adoption benefits a narrow slice of workers.

### Recommendations (actionable)
1) Rapid skills capacity (0–18 months): short-cycle credentials + employer-led apprenticeships in data, cloud, and cybersecurity.  
2) Infrastructure + affordability (6–36 months): broadband + electricity reliability + device affordability targeted to youth and women.  
3) “Small AI” for inclusion (now): deploy practical low-cost AI tools on everyday devices and pair with AI literacy + safeguards.  

### Limitations
These are global proxies for demand and supply, not a substitute for full job-posting microdata or administrative wage/occupation records. The framework is designed to be transparent, reproducible, and easy to improve as better demand-side signals become available.
